# Vacuum Cleaner - A Star (Chi phí là f(n) = g(n) + h(n))

## function A*(Start, Goal):
1. Khởi tạo tập FRONTIER = {Start} với f(Start) = g(Start) + h(Start) = 0 + h(Start)
2. Khởi tạo tập REACHED = {}
3. TRONG KHI (FRONTIER không rỗng):
    - a. Chọn trạng thái n từ FRONTIER có giá trị f(n) nhỏ nhất.
    - b. NẾU n == Goal: TRẢ VỀ "Thành công" và truy xuất lại đường đi từ Start đến n.
    - c. Loại bỏ n khỏi FRONTIER và thêm n vào REACHED.
    - d. Với mỗi trạng thái m kề với n:
        - i. Tính toán chi phí thực tế mới:

            g_new(m) = g(n) + cost(n, m)

        - ii. NẾU m đã nằm trong REACHED:

            NẾU g_new(m) >= g(m) hiện tại: Bỏ qua trạng thái m (tệ hơn).

            NGƯỢC LẠI: Xóa m khỏi REACHED và cập nhật lại g(m) = g_new(m).

        - iii. NẾU m đã nằm trong FRONTIER:

            - NẾU g_new(m) < g(m) hiện tại:

                Cập nhật lại g(m) = g_new(m) và f(m) = g(m) + h(m).

                Cập nhật lại đỉnh cha của m là n.

            iv. NẾU m chưa có mặt trong FRONTIER và REACHED:

                Gán g(m) = g_new(m). Tính f(m) = g(m) + h(m). Gán đỉnh cha của m là n. Thêm m vào FRONTIER.

    4. TRẢ VỀ "Thất bại"
    
       Không tìm thấy đường đi tới đích.

In [ ]:
import random
import copy

class Node:
    def __init__(self, floor_state, position: tuple, parent, birth_action):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.path_cost = 0 #số ô bẩn sô với GOAL
        self.heuristic_cost = 0 #khoảng cách manhattan
        self.total_cost = 0
        
    def cal_total_cost(self):
        self.total_cost = self.path_cost + self.heuristic_cost
        
    def get_tuple_floor_state(self):
        return tuple(tuple(row) for row in self.floor_state)
    
class Queue:
    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0
    
    def enqueue(self, node):
        self.queue.append(node)
        
    def dequeue(self) -> Node:
        return self.queue.pop(0)
    
    def pop(self, index: int):
        return self.queue.pop(index)
        
    def pop_the_highest_priority(self) -> Node:
        #thêm 1 phương thức pop ưu tiên cao nhất (cost thấp nhất) trong queue
        max_priority_index = 0
        for i in range(len(self.queue)):
            if self.queue[i].total_cost < self.queue[max_priority_index].total_cost:
                max_priority_index = i
        return self.queue.pop(max_priority_index)
    
    def contain(self, other: Node):
        for i, node in enumerate(self.queue):
            if node.floor_state == other.floor_state and node.position == other.position:
                return i
        return -1


class Stack:
    def __init__(self):
        self.stack = []
    def is_empty(self):
        return len(self.stack) == 0
    def push(self, node: Node):
        self.stack.append(node)
    def pop(self) -> Node:
        return self.stack.pop()
    def contain(self, other: Node):
        for node in self.stack:
            if node.floor_state == other.floor_state and node.position == other.position:
                return True
        return False

GOAL_STATE = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
ROW, COL = 3, 3

def floor_and_vacuumpos_initialize():
    floor = []
    vacuum_pos = (random.randint(0, ROW - 1), random.randint(0, COL - 1))
    for i in range(ROW):
        floor.append([random.randint(0, 1) for _ in range(COL)])
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def posible_moves(vacuum_pos):
    moves = []
    if vacuum_pos[0] > 0: moves.append("UP")
    if vacuum_pos[0] < ROW - 1: moves.append("DOWN")
    if vacuum_pos[1] > 0: moves.append("LEFT")
    if vacuum_pos[1] < COL - 1: moves.append("RIGHT")
    return moves

def apply_move(floor, vacuum_pos, move):
    temp_floor = copy.deepcopy(floor)
    if move == "UP": temp_vac_pos = (vacuum_pos[0] - 1, vacuum_pos[1])
    elif move == "DOWN": temp_vac_pos = (vacuum_pos[0] + 1, vacuum_pos[1])
    elif move == "LEFT": temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] - 1)
    else: temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] + 1)
    
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
    return temp_floor, temp_vac_pos

def get_n0_of_dirty_tiles(floor: list) -> int:
    """
    Counting the number of dirty tiles
    """
    number_of_dirty_tile = 0
    for row in floor:
        for tile in row:
            if tile == 1:
                number_of_dirty_tile += 1
    return number_of_dirty_tile

def find_dirty_tiles_position(floor) -> list:
    """
    Find all the position (x, y) of dirty tiles on the floor

    """
    positions = []
    for row in range(ROW):
        for col in range(COL):
            if floor[row][col] == 1:
                positions.append((row, col))
    return positions

def min_manhattan_distance(floor, vacuum_pos) -> int:
    """
    Find the nearest dirty tiles
    """
    dirty_tiles = find_dirty_tiles_position(floor)
    if not dirty_tiles:
        return 0
    dis = abs(dirty_tiles[0][0]-vacuum_pos[0]) + abs(dirty_tiles[0][1]-vacuum_pos[1])
    for pos in dirty_tiles:
        temp_dis = abs(pos[0]-vacuum_pos[0]) + abs(pos[1]-vacuum_pos[1])
        if dis > temp_dis:
            dis = temp_dis
    return dis
           
def run_A_Star():
    """
    Chi phí sử dụng trong bài:
        - h(n): Khoảng cách Manhattan
        - g(n): số ô dơ
    """
    floor, vacuum_pos = floor_and_vacuumpos_initialize()
    frontier = Queue()
    root = Node(floor_state=floor, position=vacuum_pos, parent=None, birth_action= None)
    root.path_cost = get_n0_of_dirty_tiles(floor)
    root.heuristic_cost = min_manhattan_distance(floor, vacuum_pos)
    root.cal_total_cost()
    frontier.enqueue(root)
    visited_state = set()
    step = 0
    
    while True:
        if frontier.is_empty(): #dừng khi frontier trống (hết cách)
            print("Frontier is empty!!")
            break
        
        current_node = frontier.pop_the_highest_priority()
        step += 1
        if (current_node.get_tuple_floor_state(), current_node.position) not in visited_state: #check nếu state node có trong visited
            visited_state.add((current_node.get_tuple_floor_state(), current_node.position))
        else:
            continue
        
        if current_node.floor_state == GOAL_STATE: #dừng khi tìm thấy đáp án (GOAL)
            return current_node
        
        moves = posible_moves(current_node.position)
        for move in moves:
            # chạy thử từng bước có thể để sinh node con
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            temp_node = Node(temp_floor, temp_vacuum_pos, current_node, move) #sinh node con
            temp_node.path_cost = get_n0_of_dirty_tiles(temp_floor) + current_node.path_cost
            temp_node.heuristic_cost = min_manhattan_distance(temp_floor, temp_vacuum_pos)
            temp_node.cal_total_cost()
                        
            if frontier.contain(temp_node)== -1 and (temp_node.get_tuple_floor_state(), temp_vacuum_pos) not in visited_state:
                #kiểm tra nếu node con không có trong visited hoặc frontier
                frontier.enqueue(temp_node)
                
            #nếu có 1 node có c ùng state nhưng cost tốt hơn thì thay thế node đó
            elif frontier.contain(temp_node) != -1:
                target_index = frontier.contain(temp_node)
                target_node = frontier.queue[target_index]
                if temp_node.total_cost < target_node.total_cost:
                    frontier.pop(target_index)
                    frontier.enqueue(temp_node)
            
                
def main():
    result = run_A_Star()
    if isinstance(result, Node):
        print(f"\nĐã tìm thấy đáp án (GOAL FOUND)!!!")
        
        # Trích xuất đường đi xuôi dòng từ Root đến Goal
        path = []
        current = result
        while current is not None:
            path.append(current)
            current = current.parent
        path.reverse()
        
        # In chi tiết từng bước di chuyển
        for step, node in enumerate(path):
            if step == 0:
                print(f"Bước xuất phát:")
            else:
                print(f"Bước {step}: Đi [{node.birth_action}]")
            for row in node.floor_state:
                print(row)
            print(f"Vị trí Robot: {node.position}")
            print("-" * 20)
    else:
        print(f"Không tìm thấy đáp án: {result}")

if __name__ == "__main__":
    main()